In [0]:
storage_account_name = "stecommercepipeline"
silver_path = f"abfss://silver@{storage_account_name}.dfs.core.windows.net/"
gold_path = f"abfss://gold@{storage_account_name}.dfs.core.windows.net/"

orders = spark.read.format("delta").load(silver_path + "orders")
items = spark.read.format("delta").load(silver_path + "order_items")
payments = spark.read.format("delta").load(silver_path + "payments")
products = spark.read.format("delta").load(silver_path + "products")
customers = spark.read.format("delta").load(silver_path + "customers")

In [0]:
from pyspark.sql.functions import sum as _sum, date_format

revenue_df = orders.join(items, "order_id") \
    .withColumn("order_month", date_format("order_purchase_timestamp", "yyyy-MM")) \
    .groupBy("order_month") \
    .agg(_sum("price").alias("total_revenue"))

revenue_df.write.format("delta").mode("overwrite").save(gold_path + "monthly_revenue")
display(revenue_df)

order_month,total_revenue
2017-12,743914.1700000198
2018-02,844178.7100000293
2017-02,247303.01999999527
2016-12,10.9
2018-07,895507.2200000228
2018-06,865124.3100000205
2017-05,506071.14000000927
2016-10,49507.66000000017
2018-08,854686.330000024
2017-04,359927.2300000009


In [0]:
category_revenue = items.join(products, "product_id") \
    .join(orders, "order_id") \
    .groupBy("product_category_name") \
    .agg(_sum("price").alias("total_sales")) \
    .orderBy("total_sales", ascending=False)

category_revenue.write.format("delta").mode("overwrite").save(gold_path + "category_sales")
display(category_revenue)

product_category_name,total_sales
beleza_saude,1258681.3399999696
relogios_presentes,1205005.6799999983
cama_mesa_banho,1036988.6800000716
esporte_lazer,988048.9700000401
informatica_acessorios,911954.3200000378
moveis_decoracao,729762.490000042
cool_stuff,635290.8500000008
utilidades_domesticas,632248.6600000213
automotivo,592720.1100000107
ferramentas_jardim,485256.4600000146


In [0]:
from pyspark.sql.functions import count

customer_orders = orders.groupBy("customer_id").agg(count("order_id").alias("order_count"))
customer_orders.write.format("delta").mode("overwrite").save(gold_path + "customer_order_counts")
display(customer_orders)

customer_id,order_count
8628fac2267e8c8804525da99c10ed0e,1
df5c9e02596851fcbfc98804e4108bbd,1
6feea03756fd7ef4fb37d8f7e44aaeb9,1
2c5c2fb51bbbcdcc297f3b37449627bb,1
8d0c2ae2c7b532f0e90bdd06bc9a04b7,1
02ed2cff54eb047cdbe79dd0535945d4,1
f06a94a401e52fb019c72f2e8bbf6a2f,1
0966fbba1c0e8c5e26e31166c2fd3ce8,1
2d2b1b42705b2f080c47fb2c46db46f6,1
b003d09f32a12bcc00b6ca04d46554e6,1
